<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-05-controllable-dispute-assistant-for-meridian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 (graded) — Controllable dispute assistant for Meridian
**Course 3: AI Agents and Agentic AI with Python — Chapter 5: Planning & control**

**Problem brief (Leo Farkas, Meridian Bank):** "A card-dispute resolution has ~8 steps,
branches, and points where a human must sign off. I need it controllable and auditable, but
still adaptive."

**What you'll submit:** a LangGraph graph with a real policy-check node, a draft node, a
genuine human-approval interrupt, a conditional escalation edge, replanning on a tool
failure, and the full audit log.

In [ ]:
!pip install -q langgraph

## 1. State, policy corpus, and node functions

In [ ]:
from typing import TypedDict, Optional

POLICY = {
    'small': 'Disputes under $500 can be resolved by automatic refund if the merchant category is low-risk.',
    'large': 'Disputes of $500 or more require escalation to a human reviewer before any resolution.',
}

class DisputeState(TypedDict):
    dispute_id: str
    amount: float
    reason: str
    policy_note: Optional[str]
    draft: Optional[str]
    approved: Optional[bool]
    resolution: Optional[str]
    audit_log: list
    tool_attempts: int

def log(state, message):
    state.setdefault('audit_log', []).append(message)
    return state

def intake(state: DisputeState) -> DisputeState:
    return log(state, f"Intake: dispute {state['dispute_id']}, ${state['amount']}, reason='{state['reason']}'")

def policy_check(state: DisputeState) -> DisputeState:
    """A real (if tiny) RAG-style lookup: pick the policy clause that applies to this amount."""
    state['tool_attempts'] = state.get('tool_attempts', 0) + 1
    if state['tool_attempts'] == 1 and state['dispute_id'] == 'DSP-FAIL-DEMO':
        # simulate a transient tool failure to demonstrate replanning
        log(state, 'Policy lookup FAILED (simulated transient error) — will retry.')
        return state
    note = POLICY['small'] if state['amount'] < 500 else POLICY['large']
    state['policy_note'] = note
    return log(state, f'Policy check: {note}')

def draft_resolution(state: DisputeState) -> DisputeState:
    draft = f"Proposed resolution for {state['dispute_id']}: refund ${state['amount']} per policy."
    state['draft'] = draft
    return log(state, f'Drafted: {draft}')

def resolve(state: DisputeState) -> DisputeState:
    state['resolution'] = state['draft'] if state.get('approved') else 'HELD — awaiting human review'
    return log(state, f"Resolved: {state['resolution']}")

## 2. Build the graph with a conditional escalation edge and a retry edge

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def route_after_policy(state: DisputeState) -> str:
    if state.get('policy_note') is None:
        return 'policy_check'  # replan: the tool failed, try again
    return 'escalate' if state['amount'] >= 500 else 'draft_resolution'

def escalate(state: DisputeState) -> DisputeState:
    return log(state, f"Escalated: ${state['amount']} exceeds the auto-resolve threshold.")

graph = StateGraph(DisputeState)
graph.add_node('intake', intake)
graph.add_node('policy_check', policy_check)
graph.add_node('escalate', escalate)
graph.add_node('draft_resolution', draft_resolution)
graph.add_node('human_approval', lambda state: log(state, 'Awaiting human approval...'))
graph.add_node('resolve', resolve)

graph.set_entry_point('intake')
graph.add_edge('intake', 'policy_check')
graph.add_conditional_edges('policy_check', route_after_policy,
                             {'policy_check': 'policy_check', 'escalate': 'escalate', 'draft_resolution': 'draft_resolution'})
graph.add_edge('escalate', 'draft_resolution')
graph.add_edge('draft_resolution', 'human_approval')
graph.add_edge('human_approval', 'resolve')
graph.add_edge('resolve', END)

checkpointer = MemorySaver()
# the REAL human-in-the-loop mechanism: the graph physically pauses before this node
app = graph.compile(checkpointer=checkpointer, interrupt_before=['human_approval'])
print('Graph compiled with a hard interrupt before human_approval.')

## 3. Run it: the graph genuinely pauses for approval

In [ ]:
config = {'configurable': {'thread_id': 'dispute-001'}}
initial_state = {'dispute_id': 'DSP-001', 'amount': 620.0, 'reason': 'item not received', 'audit_log': [], 'tool_attempts': 0}

result = app.invoke(initial_state, config)
print('State after first invoke (paused before human_approval):')
print(' resolution:', result.get('resolution'), '(should be None — the graph has not resumed)')
for line in result['audit_log']:
    print(' -', line)

assert result.get('resolution') is None, 'The graph should be paused, not resolved, at this point.'
print('\nConfirmed: the resolution literally cannot execute until a human approves.')

In [ ]:
# The human reviews the audit log above and approves — this is a REAL, separate step,
# not something the model can do to itself
app.update_state(config, {'approved': True})
final_result = app.invoke(None, config)  # resume from the checkpoint
print('Final resolution:', final_result['resolution'])
print('\nFull audit log:')
for line in final_result['audit_log']:
    print(' -', line)

## 4. Replanning on a tool failure

In [ ]:
fail_config = {'configurable': {'thread_id': 'dispute-fail-demo'}}
fail_state = {'dispute_id': 'DSP-FAIL-DEMO', 'amount': 120.0, 'reason': 'duplicate charge', 'audit_log': [], 'tool_attempts': 0}

fail_result = app.invoke(fail_state, fail_config)
print('Audit log showing the retry after a simulated tool failure:')
for line in fail_result['audit_log']:
    print(' -', line)
assert any('FAILED' in l for l in fail_result['audit_log']) and any('Policy check:' in l for l in fail_result['audit_log']), \
    'Expected both a failure log line and a successful retry log line.'
print('\nReplanning confirmed: the graph retried policy_check after the simulated failure.')

## 5. Model-routed vs. hard-coded (fill in)
List which decisions in this graph are hard-coded (the edges you wrote) vs. where you'd
actually want a model making the call (e.g. drafting the resolution text itself, or judging
whether a dispute reason is plausible). Why does keeping the *routing* hard-coded matter for
a regulated process like this, even though the *content* generation can still be model-driven?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 5: Planning & control*